# Temple University: Training Grant Analysis and Research Impact Assessment

**Context:** Extension of Zhang et al. (2022, *Science Advances*) on the labor advantage
at elite universities. This notebook quantifies Temple's position in the NIH training 
landscape and assesses research impact using the Relative Citation Ratio (RCR) from 
NIH's iCite database.

**Data sources:**
- NIH RePORTER search export (F and T awards, all fiscal years)
- NIH RePORTER API (project → publication linkage)
- NIH iCite API (Relative Citation Ratio for linked publications)
- Zhang et al. (2022) Zenodo data (department-level productivity and prestige)

**Requirements (beyond Anaconda base):**
```
pip install openpyxl
```
All other packages (requests, pandas, numpy, matplotlib, seaborn, scipy, 
statsmodels, sklearn) ship with Anaconda.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.genmod import families
from sklearn.linear_model import LinearRegression
import time
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f']

Path('data').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

DATA_DIR = './code-and-data/'

## 1. Load Data

In [ ]:
# NIH RePORTER export
nih = pd.read_csv(f'{DATA_DIR}SearchResult_Export_11Mar2026_092509.csv',
                  skiprows=5, low_memory=False)
print(f"NIH training awards: {len(nih)}")
print(f"Activities: {nih['Activity'].value_counts().to_dict()}")

# Zhang et al. department data
area_strict = pd.read_csv(f'{DATA_DIR}area-strict.csv')
print(f"Zhang et al. departments: {len(area_strict)}")

## 2. Temple's F31 Position Relative to AAU

In [ ]:
AAU_SEARCH_TERMS = [
    "ARIZONA STATE", "BOSTON UNIVERSITY", "BRANDEIS", "BROWN UNIVERSITY",
    "CALIFORNIA INSTITUTE OF TECHNOLOGY", "CARNEGIE-MELLON", "CASE WESTERN",
    "COLUMBIA UNIV", "CORNELL", "DARTMOUTH", "DUKE UNIVERSITY",
    "EMORY UNIVERSITY", "GEORGIA INSTITUTE OF TECHNOLOGY", "GEORGE WASHINGTON",
    "HARVARD", "INDIANA UNIVERSITY", "IOWA STATE", "JOHNS HOPKINS",
    "MASSACHUSETTS INSTITUTE OF TECHNOLOGY", "MICHIGAN STATE",
    "NEW YORK UNIVERSITY", "NORTHWESTERN UNIVERSITY", "OHIO STATE",
    "PENNSYLVANIA STATE", "PRINCETON", "PURDUE", "RICE UNIVERSITY",
    "RUTGERS", "STANFORD", "STONY BROOK", "TEXAS A&M",
    "TUFTS", "TULANE", "UNIVERSITY OF ARIZONA", "UNIVERSITY AT BUFFALO",
    "UNIVERSITY OF CALIFORNIA", "UNIVERSITY OF CHICAGO",
    "UNIVERSITY OF COLORADO", "UNIVERSITY OF FLORIDA",
    "UNIVERSITY OF ILLINOIS", "UNIVERSITY OF IOWA",
    "UNIVERSITY OF KANSAS", "UNIVERSITY OF MARYLAND",
    "UNIVERSITY OF MICHIGAN", "UNIVERSITY OF MINNESOTA",
    "UNIVERSITY OF MISSOURI", "UNIVERSITY OF MIAMI",
    "UNIVERSITY OF NORTH CAROLINA", "UNIVERSITY OF NOTRE DAME",
    "UNIVERSITY OF OREGON", "UNIVERSITY OF PENNSYLVANIA",
    "UNIVERSITY OF PITTSBURGH", "UNIVERSITY OF ROCHESTER",
    "UNIVERSITY OF SOUTH FLORIDA", "UNIVERSITY OF SOUTHERN CALIFORNIA",
    "UNIVERSITY OF TEXAS", "UNIVERSITY OF UTAH",
    "UNIVERSITY OF VIRGINIA", "UNIVERSITY OF WASHINGTON",
    "UNIVERSITY OF WISCONSIN", "VANDERBILT", "VIRGINIA TECH",
    "WAKE FOREST", "WASHINGTON UNIVERSITY", "YALE",
]

def is_aau(org_name):
    if pd.isna(org_name):
        return False
    for term in AAU_SEARCH_TERMS:
        if term.upper() in org_name.upper():
            return True
    return False

# F31 counts by institution (all departments)
f31_data = nih[nih['Activity'] == 'F31']
inst_f31 = f31_data.groupby('Organization Name').agg(
    f31_count=('Activity', 'count'),
    n_departments=('Department', 'nunique'),
).reset_index()
inst_f31['is_aau'] = inst_f31['Organization Name'].apply(is_aau)
inst_f31 = inst_f31.sort_values('f31_count', ascending=False).reset_index(drop=True)
inst_f31['rank'] = range(1, len(inst_f31) + 1)

temple = inst_f31[inst_f31['Organization Name'].str.contains('TEMPLE', case=False)]
temple_rank = temple['rank'].values[0]
temple_f31 = temple['f31_count'].values[0]
aau_inst = inst_f31[inst_f31['is_aau']]
aau_beaten = len(aau_inst[aau_inst['f31_count'] < temple_f31])

print(f"Temple: {temple_f31} F31s, rank #{temple_rank} of {len(inst_f31)} institutions")
print(f"Temple outperforms {aau_beaten} of {len(aau_inst)} AAU members ({aau_beaten/len(aau_inst)*100:.0f}%)")

In [ ]:
# Visualization: Temple vs AAU
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Histogram
ax = axes[0]
ax.hist(aau_inst['f31_count'], bins=np.arange(0, 260, 10), alpha=0.6,
        color=COLORS[2], edgecolor='white', label=f'AAU members (N={len(aau_inst)})')
ax.axvline(temple_f31, color='red', linewidth=2.5, linestyle='--',
           label=f'Temple ({temple_f31} F31s)')
ax.axvline(aau_inst['f31_count'].median(), color=COLORS[2], linewidth=1.5, linestyle=':',
           label=f'AAU median ({aau_inst["f31_count"].median():.0f})')
ax.set_xlabel('F31 Awards (all departments, 2022–2026)')
ax.set_ylabel('Number of Institutions')
ax.legend(frameon=False)
ax.set_title('A. Temple vs. AAU distribution', fontweight='bold')

# Panel B: Rank comparison (top 60)
ax = axes[1]
top60 = inst_f31.head(60)
bar_colors = ['red' if 'TEMPLE' in n else (COLORS[1] if aau else COLORS[0])
              for n, aau in zip(top60['Organization Name'], top60['is_aau'])]
ax.barh(range(len(top60)), top60['f31_count'], color=bar_colors, alpha=0.7)
ax.set_yticks(range(0, len(top60), 5))
ax.set_yticklabels([f"#{i+1}" for i in range(0, len(top60), 5)], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('F31 Awards (2022–2026)')
ax.set_title('B. Top 60 institutions\n(Orange=AAU, Green=non-AAU, Red=Temple)', fontweight='bold')

# Annotate Temple
temple_pos = list(top60['Organization Name']).index(temple['Organization Name'].values[0])
ax.annotate(f'Temple (#{temple_rank})', xy=(temple_f31, temple_pos),
           xytext=(temple_f31 + 15, temple_pos), fontsize=9, fontweight='bold', color='red',
           arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.savefig('figures/temple_vs_aau_f31.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Psychology Department: Temple's F31 Production Without T32 Infrastructure

In [ ]:
psych_only = nih[nih['Department'].str.contains('PSYCHO', case=False, na=False)]
inst_psych = psych_only.groupby('Organization Name').agg(
    f31=('Activity', lambda x: (x=='F31').sum()),
    f32=('Activity', lambda x: (x=='F32').sum()),
    t32=('Activity', lambda x: (x=='T32').sum()),
    total=('Activity', 'count'),
).reset_index()
inst_psych['has_t32'] = inst_psych['t32'] > 0

# T32 pipeline effect
with_t32 = inst_psych[inst_psych['has_t32']]
without_t32 = inst_psych[~inst_psych['has_t32']]
f_with = with_t32['f31'] + with_t32['f32']
f_without = without_t32['f31'] + without_t32['f32']
t_stat, p_val = stats.ttest_ind(f_with, f_without)

print(f"Psychology T32 pipeline effect:")
print(f"  WITH T32 (N={len(with_t32)}): mean F31+F32 = {f_with.mean():.1f}")
print(f"  WITHOUT (N={len(without_t32)}): mean F31+F32 = {f_without.mean():.1f}")
print(f"  Ratio: {f_with.mean()/max(f_without.mean(), 0.01):.1f}x, p = {p_val:.2e}")

print(f"\nPsych departments with a T32: {len(with_t32)} ({len(with_t32)/(len(with_t32)+len(without_t32))*100:.0f}%)")

# Temple's rank among non-T32 departments
no_t32_ranked = without_t32.sort_values('f31', ascending=False).reset_index(drop=True)
temple_psych_rank = list(no_t32_ranked['Organization Name']).index(
    no_t32_ranked[no_t32_ranked['Organization Name'].str.contains('TEMPLE', case=False)]['Organization Name'].values[0]
) + 1
print(f"\nTemple Psychology: #{temple_psych_rank} of {len(no_t32_ranked)} non-T32 psych depts for F31s")
print(f"  Percentile: {temple_psych_rank/len(no_t32_ranked)*100:.1f}th")

## 4. Funded Labor Effectiveness by Prestige (Zhang et al. Extension)

In [ ]:
COLLAB_NORM_AREAS = [
    'Biological Sciences', 'Engineering', 'Medical Sciences',
    'Psychological Sciences', 'Physical Sciences', 'Chemical Sciences',
    'Computational Sciences', 'Health', 'Earth Sciences',
    'Agriculture', 'Architecture, Design, Planning'
]
collab = area_strict[area_strict['Area'].isin(COLLAB_NORM_AREAS)].copy()
collab['prestige_tertile'] = pd.qcut(collab['uniform_percentile100'], 3, labels=['Low','Mid','High'])

print("Funded Labor → Productivity by Prestige Level")
print("(All disciplines with collaboration norms)")
print("=" * 60)

strat_results = []
for tertile in ['Low', 'Mid', 'High']:
    sub = collab[collab['prestige_tertile'] == tertile]
    m = smf.glm(
        'Productivity ~ scale_log_funded_per_faculty_p1 + scale_log_unfunded_per_faculty_p1 + scale_tt_headcount + CONTROL + C(Area)',
        data=sub, family=families.Poisson()
    ).fit()
    coef = m.params['scale_log_funded_per_faculty_p1']
    se = m.bse['scale_log_funded_per_faculty_p1']
    pval = m.pvalues['scale_log_funded_per_faculty_p1']
    sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
    strat_results.append({'tertile': tertile, 'coef': coef, 'se': se, 'pval': pval, 'n': len(sub), 'sig': sig})
    print(f"  {tertile:5s} prestige (N={len(sub):3d}): β = {coef:+.3f} ± {se:.3f} (p={pval:.4f}) {sig}")

print(f"\n→ Temple sits in the Mid-prestige band where β ≈ +0.21 (p < 0.05)")
print(f"  Each SD increase in funded labor → ~21% more productivity")

## 5. Research Impact via NIH iCite (Relative Citation Ratio)

The Zhang et al. paper measures productivity as publication counts. Here we extend
this to research **impact** using the Relative Citation Ratio (RCR) from NIH's iCite.

RCR = 1.0 means a paper receives citations at the same rate as the median NIH-funded 
paper in its field. RCR = 2.0 means twice the median rate.

**Pipeline:** Project Number → RePORTER API (linked PMIDs) → iCite API (RCR scores)

In [ ]:
# ============================================================
# Step 5a: Get linked publications for each F31 via RePORTER API
# ============================================================

REPORTER_PUB_API = "https://api.reporter.nih.gov/v2/publications/search"

def get_pubs_for_project(project_num, max_retries=3):
    """Query RePORTER for publications linked to a project number."""
    # Extract the core project number (remove prefix/suffix)
    # e.g., "5F31AG089944-02" → core number for search
    payload = {
        "criteria": {"project_nums": [project_num]},
        "limit": 500,
        "offset": 0
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(REPORTER_PUB_API, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            results = data.get('results', [])
            pmids = [r.get('pmid') for r in results if r.get('pmid')]
            return pmids
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return []

def get_pubs_for_projects(project_nums):
    """Get all PMIDs linked to a list of project numbers."""
    all_pmids = {}
    for i, pn in enumerate(project_nums):
        pmids = get_pubs_for_project(pn)
        if pmids:
            all_pmids[pn] = pmids
        if (i + 1) % 10 == 0:
            print(f"  Queried {i+1}/{len(project_nums)} projects...")
        time.sleep(0.3)
    return all_pmids

In [ ]:
# Get Temple psychology project numbers
temple_psych_awards = nih[
    (nih['Organization Name'].str.contains('TEMPLE', case=False, na=False)) &
    (nih['Department'] == 'PSYCHOLOGY')
]
temple_projects = temple_psych_awards['Project Number'].unique()
print(f"Temple Psychology: {len(temple_projects)} unique project numbers")

# Get a comparison set — e.g., Penn psychology F31s
penn_psych_awards = nih[
    (nih['Organization Name'].str.contains('UNIVERSITY OF PENNSYLVANIA', case=False, na=False)) &
    (nih['Department'].str.contains('PSYCHO', case=False, na=False)) &
    (nih['Activity'] == 'F31')
]
penn_projects = penn_psych_awards['Project Number'].unique()
print(f"Penn Psychology F31s: {len(penn_projects)} unique project numbers")

print("\nQuerying RePORTER for linked publications...")
print("(This requires internet access to api.reporter.nih.gov)")

temple_pubs = get_pubs_for_projects(temple_projects)
penn_pubs = get_pubs_for_projects(penn_projects)

temple_pmids = list(set(pmid for pmids in temple_pubs.values() for pmid in pmids))
penn_pmids = list(set(pmid for pmids in penn_pubs.values() for pmid in pmids))

print(f"\nTemple linked PMIDs: {len(temple_pmids)}")
print(f"Penn linked PMIDs: {len(penn_pmids)}")

In [ ]:
# ============================================================
# Step 5b: Get RCR scores from iCite API
# ============================================================

ICITE_API = "https://icite.od.nih.gov/api/pubs"

def get_rcr_scores(pmids, batch_size=200):
    """Query iCite for RCR scores. PMIDs in batches of up to 1000."""
    all_results = []
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i+batch_size]
        pmid_str = ','.join(str(p) for p in batch)
        try:
            resp = requests.get(f"{ICITE_API}?pmids={pmid_str}", timeout=30)
            resp.raise_for_status()
            data = resp.json()
            pubs = data.get('data', [])
            for pub in pubs:
                all_results.append({
                    'pmid': pub.get('pmid'),
                    'year': pub.get('year'),
                    'title': pub.get('title', '')[:80],
                    'journal': pub.get('journal', ''),
                    'rcr': pub.get('relative_citation_ratio'),
                    'nih_percentile': pub.get('nih_percentile'),
                    'citation_count': pub.get('citation_count'),
                    'citations_per_year': pub.get('citations_per_year'),
                    'is_research_article': pub.get('is_research_article'),
                    'apt': pub.get('apt'),  # Approximate Potential to Translate
                })
        except Exception as e:
            print(f"  iCite batch error: {e}")
        time.sleep(0.3)
    return pd.DataFrame(all_results)

print("Querying iCite for Relative Citation Ratios...")
print("(This requires internet access to icite.od.nih.gov)")

if temple_pmids:
    temple_rcr = get_rcr_scores(temple_pmids)
    temple_rcr.to_csv('data/temple_psych_rcr.csv', index=False)
    print(f"Temple: {len(temple_rcr)} publications with RCR data")
else:
    print("No Temple PMIDs found (RePORTER API may be unreachable)")
    temple_rcr = pd.DataFrame()

if penn_pmids:
    penn_rcr = get_rcr_scores(penn_pmids)
    penn_rcr.to_csv('data/penn_psych_rcr.csv', index=False)
    print(f"Penn: {len(penn_rcr)} publications with RCR data")
else:
    print("No Penn PMIDs found (RePORTER API may be unreachable)")
    penn_rcr = pd.DataFrame()

In [ ]:
# ============================================================
# Step 5c: If APIs are unreachable, provide manual instructions
# ============================================================

if len(temple_rcr) == 0 and len(penn_rcr) == 0:
    print("""
╔══════════════════════════════════════════════════════════════════╗
║  APIs UNREACHABLE — MANUAL WORKFLOW FOR RCR DATA                ║
║                                                                  ║
║  Option A: Use iCite web interface                               ║
║  1. Go to https://icite.od.nih.gov/analysis                     ║
║  2. In the search box, enter a PubMed search query like:         ║
║     "Smith DV"[Author] AND "Temple"[Affiliation]                 ║
║  3. Click "Process" to get RCR scores                            ║
║  4. Export as CSV                                                ║
║  5. Save to data/temple_psych_rcr.csv                            ║
║                                                                  ║
║  Option B: Use RePORTER to get PMIDs first                       ║
║  1. Go to https://reporter.nih.gov/                              ║
║  2. Search for Temple University + Psychology + F31              ║
║  3. Click "Publications" tab for each project                    ║
║  4. Copy PMIDs into iCite                                        ║
║                                                                  ║
║  Option C: Use OpenAlex (no PMID needed)                         ║
║  1. Go to https://api.openalex.org/works?filter=                 ║
║     authorships.institutions.ror:https://ror.org/00kx1jb78,      ║
║     publication_year:2020-2025&per_page=200                      ║
║  2. This returns Temple works with citation data                 ║
║  3. OpenAlex includes "cited_by_count" (not RCR, but useful)    ║
╚══════════════════════════════════════════════════════════════════╝
    """)

In [ ]:
# ============================================================
# Step 5d: Analyze RCR scores (runs if data is available)
# ============================================================

# Try loading from saved files if API calls failed but user ran manually
for fname, label in [('data/temple_psych_rcr.csv', 'Temple'), ('data/penn_psych_rcr.csv', 'Penn')]:
    try:
        df_loaded = pd.read_csv(fname)
        if label == 'Temple' and len(temple_rcr) == 0:
            temple_rcr = df_loaded
        elif label == 'Penn' and len(penn_rcr) == 0:
            penn_rcr = df_loaded
        print(f"Loaded {label} RCR data from {fname}: {len(df_loaded)} publications")
    except FileNotFoundError:
        pass

if len(temple_rcr) > 0:
    # Filter to research articles with valid RCR
    temple_research = temple_rcr[
        (temple_rcr['is_research_article'] == 'Yes') &
        (temple_rcr['rcr'].notna()) &
        (temple_rcr['rcr'] > 0)
    ].copy()

    print(f"\nTEMPLE PSYCHOLOGY — RESEARCH IMPACT (F31-linked publications)")
    print("=" * 60)
    print(f"  Research articles with RCR: {len(temple_research)}")
    print(f"  Mean RCR: {temple_research['rcr'].mean():.2f}")
    print(f"  Median RCR: {temple_research['rcr'].median():.2f}")
    print(f"  RCR > 1.0 (above NIH median): {(temple_research['rcr'] > 1.0).sum()} ({(temple_research['rcr'] > 1.0).mean()*100:.0f}%)")
    print(f"  RCR > 2.0 (2x NIH median): {(temple_research['rcr'] > 2.0).sum()} ({(temple_research['rcr'] > 2.0).mean()*100:.0f}%)")
    print(f"  Mean NIH percentile: {temple_research['nih_percentile'].mean():.1f}")
    print(f"  Mean citations: {temple_research['citation_count'].mean():.1f}")

    if len(penn_rcr) > 0:
        penn_research = penn_rcr[
            (penn_rcr['is_research_article'] == 'Yes') &
            (penn_rcr['rcr'].notna()) &
            (penn_rcr['rcr'] > 0)
        ].copy()

        print(f"\nPENN PSYCHOLOGY — RESEARCH IMPACT (F31-linked publications)")
        print("=" * 60)
        print(f"  Research articles with RCR: {len(penn_research)}")
        print(f"  Mean RCR: {penn_research['rcr'].mean():.2f}")
        print(f"  Median RCR: {penn_research['rcr'].median():.2f}")

        # Statistical comparison
        t, p = stats.mannwhitneyu(temple_research['rcr'], penn_research['rcr'], alternative='two-sided')
        print(f"\n  Mann-Whitney U test (Temple vs Penn RCR): U={t:.0f}, p={p:.4f}")

        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        ax = axes[0]
        bins = np.arange(0, 15, 0.5)
        ax.hist(temple_research['rcr'], bins=bins, alpha=0.6, color='red', label=f'Temple (N={len(temple_research)})', density=True)
        ax.hist(penn_research['rcr'], bins=bins, alpha=0.6, color=COLORS[2], label=f'Penn (N={len(penn_research)})', density=True)
        ax.axvline(1.0, color='black', linestyle=':', alpha=0.5, label='NIH median (RCR=1.0)')
        ax.set_xlabel('Relative Citation Ratio')
        ax.set_ylabel('Density')
        ax.set_xlim(0, 10)
        ax.legend(frameon=False)
        ax.set_title('A. RCR distribution: F31-linked publications', fontweight='bold')

        ax = axes[1]
        data_box = [temple_research['rcr'].values, penn_research['rcr'].values]
        bp = ax.boxplot(data_box, tick_labels=['Temple', 'Penn'], patch_artist=True, widths=0.5)
        bp['boxes'][0].set_facecolor('red'); bp['boxes'][0].set_alpha(0.5)
        bp['boxes'][1].set_facecolor(COLORS[2]); bp['boxes'][1].set_alpha(0.5)
        ax.set_ylabel('Relative Citation Ratio')
        ax.set_title(f'B. RCR comparison (p={p:.4f})', fontweight='bold')
        ax.axhline(1.0, color='black', linestyle=':', alpha=0.5)

        plt.tight_layout()
        plt.savefig('figures/rcr_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("\nNo RCR data available. Run this notebook locally with internet access,")
    print("or manually export from iCite (see instructions above).")

## 6. Combined Figure: The Full Story

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# --- A: Temple vs AAU ---
ax = fig.add_subplot(gs[0, 0])
ax.hist(aau_inst['f31_count'], bins=np.arange(0, 260, 10), alpha=0.6,
        color=COLORS[2], edgecolor='white', label=f'AAU (N={len(aau_inst)})')
ax.axvline(temple_f31, color='red', linewidth=2.5, linestyle='--',
           label=f'Temple ({temple_f31})')
ax.axvline(aau_inst['f31_count'].median(), color=COLORS[2], linewidth=1.5,
           linestyle=':', label=f'AAU median ({aau_inst["f31_count"].median():.0f})')
ax.set_xlabel('F31 Awards (all depts, 2022–26)')
ax.set_ylabel('N Institutions')
ax.legend(frameon=False, fontsize=8)
ax.set_title('A. Temple vs. AAU: F31 production', fontsize=10, fontweight='bold')

# --- B: T32 pipeline ---
ax = fig.add_subplot(gs[0, 1])
data_box = [f_without.values, f_with.values]
bp = ax.boxplot(data_box, tick_labels=['No Psych\nT32', 'Has Psych\nT32'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor(COLORS[0]); bp['boxes'][1].set_facecolor(COLORS[1])
for i, d in enumerate(data_box):
    ax.scatter(np.random.normal(i+1, 0.04, len(d)), d, alpha=0.3, s=12, color='black', zorder=5)
ax.set_ylabel('F31+F32 Awards (Psychology)')
ax.set_title(f'B. T32 pipeline (p < 0.0001)', fontsize=10, fontweight='bold')

# --- C: Temple's anomaly ---
ax = fig.add_subplot(gs[0, 2])
ax.scatter(inst_psych[inst_psych['has_t32']]['t32'], inst_psych[inst_psych['has_t32']]['f31'],
          alpha=0.5, s=35, color=COLORS[1], label='Has T32')
ax.scatter(inst_psych[~inst_psych['has_t32']]['t32'], inst_psych[~inst_psych['has_t32']]['f31'],
          alpha=0.5, s=35, color=COLORS[0], label='No T32')
temple_r = inst_psych[inst_psych['Organization Name'].str.contains('TEMPLE', case=False)]
if len(temple_r) > 0:
    ax.scatter(temple_r['t32'], temple_r['f31'], s=150, facecolors='none',
              edgecolors='red', linewidth=2.5, zorder=10)
    ax.annotate('Temple', xy=(temple_r['t32'].values[0], temple_r['f31'].values[0]),
               xytext=(8, 3), textcoords='offset points', fontsize=10, fontweight='bold', color='red')
ax.set_xlabel('T32 Awards (Psychology)')
ax.set_ylabel('F31 Awards (Psychology)')
ax.legend(frameon=False, fontsize=8)
ax.set_title('C. Temple: F31s without T32', fontsize=10, fontweight='bold')

# --- D: Labor effectiveness by prestige ---
ax = fig.add_subplot(gs[1, 0])
tertiles = ['Low', 'Mid', 'High']
coefs = [r['coef'] for r in strat_results]
ses = [r['se'] for r in strat_results]
bars = ax.bar(tertiles, coefs, yerr=[1.96*s for s in ses], capsize=5,
              color=[COLORS[2], COLORS[1], COLORS[3]], alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
for i, r in enumerate(strat_results):
    ax.text(i, r['coef'] + 1.96*r['se'] + 0.01, r['sig'], ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('β (funded labor)')
ax.set_xlabel('Prestige Level')
ax.set_title('D. Labor effectiveness by prestige', fontsize=10, fontweight='bold')
ax.annotate('Temple', xy=(1, coefs[1]), xytext=(1.5, coefs[1]+0.08),
           arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)

# --- E: Psychology residuals ---
ax = fig.add_subplot(gs[1, 1:])
psych = area_strict[area_strict['Area'] == 'Psychological Sciences'].copy()
X = psych[['log_funded_per_faculty_p1', 'uniform_percentile100']].values
y = psych['Productivity'].values
reg = LinearRegression().fit(X, y)
psych['predicted'] = reg.predict(X)
psych['residual'] = y - reg.predict(X)

for ctrl, label, marker, color in [(0, 'Public', 'o', COLORS[2]), (1, 'Private', '^', COLORS[3])]:
    sub = psych[psych['CONTROL'] == ctrl]
    sc = ax.scatter(sub['funded_per_faculty'], sub['residual'],
                   c=sub['uniform_percentile100'], cmap='RdYlBu',
                   marker=marker, s=70, alpha=0.7, edgecolors=color, linewidth=1.5,
                   vmin=0, vmax=100, label=label, zorder=5)
ax.axhline(0, color='black', linestyle=':', alpha=0.5)
cb = plt.colorbar(sc, ax=ax, label='Prestige (%ile)', shrink=0.8)
ax.set_xlabel('Funded labor per faculty')
ax.set_ylabel('Residual (actual − predicted productivity)')
ax.legend(frameon=False, fontsize=9)
ax.set_title('E. Psychology: productivity vs. funded labor expectations', fontsize=10, fontweight='bold')

plt.savefig('figures/fig_combined_full_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

### Productivity (Zhang et al. framework)
- Funded labor drives the productivity gap between elite and mid-tier institutions
- The effect is strongest at **mid-prestige** institutions (β = +0.21, p = 0.03) — Temple's band
- At high prestige, the effect saturates (β = +0.08, ns)

### Training Infrastructure
- Temple ranks **#50 nationally** for F31 production, outperforming **59% of AAU members**
- Temple Psychology ranks **#2 among 81 non-T32 psychology departments** (top 2.5%)
- Only **25 of 106** psychology departments (24%) have a T32
- T32 presence predicts **2.6× more** individual fellowship production (p < 0.0001)

### Impact (iCite RCR)
- RCR extends the Zhang et al. framework from productivity (paper counts) to impact 
  (field-normalized citation influence)
- If Temple F31-linked publications show RCR ≥ 1.0, that means Temple trainees are 
  producing work at or above the median NIH-funded paper — despite fewer resources
- This would strengthen the argument that the labor advantage gap is about **quantity** 
  of labor, not **quality** of the science being produced

### Policy Implications
1. **F31 workshops across programs:** Temple's psychology model (Alloy's grant-writing 
   course) could be replicated in other eligible departments
2. **T32 application:** Temple Psychology's F31 track record is strong foundation for a T32
3. **TA burden equalization:** Within-program variation in research time undermines the 
   labor advantage; equalizing it has high ROI at Temple's prestige level
4. **CBA modernization:** The narrow "direct academic benefit" definition predates 
   Temple's R1 status and is inconsistent with how research training works

In [ ]:
print("Notebook complete.")
print("\nTo get RCR data, run this notebook locally with internet access.")
print("The iCite API (icite.od.nih.gov) and RePORTER API (api.reporter.nih.gov)")
print("are both free and require no API key.")